In [1]:
import sys
import subprocess
import os

print("Đang cài đặt thư viện... Vui lòng đợi.")

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.46.1"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "protobuf<4"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "bitsandbytes", "peft", "accelerate", "datasets"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "sacrebleu", "evaluate", "jiwer", "nltk", "scipy"])

print("Cài đặt HOÀN TẤT.")
print("QUAN TRỌNG: HÃY KHỞI ĐỘNG LẠI KERNEL NGAY BÂY GIỜ.")

Đang cài đặt thư viện... Vui lòng đợi.
Cài đặt HOÀN TẤT.
QUAN TRỌNG: HÃY KHỞI ĐỘNG LẠI KERNEL NGAY BÂY GIỜ.


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import evaluate
import nltk
import gc

BASE_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER_PATH = "/kaggle/input/vlsp-finaltest/qwen_mt_en_vi/final_model"
TEST_SRC_FILE = "/kaggle/input/vlsp-medical-data-test/public_test.en.txt"
TEST_REF_FILE = "/kaggle/input/vlsp-medical-data-test/public_test.vi.txt"

BATCH_SIZE = 32  
MAX_NEW_TOKENS = 256 

print(">>> Đang load Model & Tokenizer...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" 

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

model.config.use_cache = True 

def read_lines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]

src_lines = read_lines(TEST_SRC_FILE)
ref_lines = read_lines(TEST_REF_FILE)

def smart_batch_translate(lines, batch_size):
    prompts = []
    system_msg = "You are a professional medical translator. Translate the following English medical text to Vietnamese accurately, preserving terminology."
    
    print("Đang chuẩn bị Prompts...")
    for text in lines:
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": text}
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt)

    lengths = [len(p) for p in prompts]
    
    sorted_indices = np.argsort(lengths)
    
    sorted_prompts = [prompts[i] for i in sorted_indices]
    
    results = []
    print(f"Bắt đầu dịch {len(lines)} câu với Batch Size = {batch_size}...")
    
    for i in tqdm(range(0, len(sorted_prompts), batch_size)):
        batch = sorted_prompts[i : i + batch_size]
        
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=1500).to("cuda:0")
        
        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,   
                num_beams=1,       
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True     
            )
        
        decoded_batch = tokenizer.batch_decode(generated_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)
        results.extend(decoded_batch)
        
        del inputs, generated_ids
        torch.cuda.empty_cache()

    final_predictions = [None] * len(lines)
    for idx, original_idx in enumerate(sorted_indices):
        final_predictions[original_idx] = results[idx].strip()
        
    return final_predictions

pred_lines = smart_batch_translate(src_lines, batch_size=BATCH_SIZE)

print(">>> Đang tính BLEU/METEOR...")
bleu = evaluate.load("sacrebleu")
meteor = evaluate.load("meteor")
ter = evaluate.load("ter")

bleu_score = bleu.compute(predictions=pred_lines, references=ref_lines)
meteor_score = meteor.compute(predictions=pred_lines, references=ref_lines)
ter_score = ter.compute(predictions=pred_lines, references=ref_lines)

print("\n" + "="*40)
print(f"KẾT QUẢ ĐÁNH GIÁ (Qwen 0.5B - Optimized):")
print(f"BLEU:   {bleu_score['score']:.4f}")
print(f"METEOR: {meteor_score['meteor']:.4f}")
print(f"TER:    {ter_score['score']:.4f}")
print("="*40 + "\n")

df = pd.DataFrame({
    "Source": src_lines,
    "Reference": ref_lines,
    "Prediction": pred_lines
})
df.to_excel("vlsp_result_optimized.xlsx", index=False)
print("Xong! File kết quả: vlsp_result_optimized.xlsx")

2025-12-23 14:56:43.722945: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766501804.174931     132 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766501804.293083     132 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766501805.437058     132 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766501805.437097     132 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766501805.437099     132 computation_placer.cc:177] computation placer alr

>>> Đang load Model & Tokenizer...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  0%|          | 0/94 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


>>> Đang tính BLEU/METEOR...


[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...



KẾT QUẢ ĐÁNH GIÁ (Qwen 0.5B - Optimized):
BLEU:   38.9080
METEOR: 0.6399
TER:    55.4144

Xong! File kết quả: vlsp_result_optimized.xlsx


In [3]:
import pandas as pd
import evaluate

try:
    df = pd.read_excel("/kaggle/input/vlsp-result-optimized/vlsp_result_optimized.xlsx")
    print(">>> Đã tìm thấy file kết quả cũ. Đang load dữ liệu...")
    
    pred_lines = df["Prediction"].astype(str).tolist()
    ref_lines = df["Reference"].astype(str).tolist()
    src_lines = df["Source"].astype(str).tolist()

    print(">>> Đang load lại thư viện đánh giá...")
    bleu = evaluate.load("sacrebleu")

    print(">>> Đang thực hiện phân tích lỗi chi tiết...")
    sentence_bleu_scores = []
    for pred, ref in zip(pred_lines, ref_lines):
        score = bleu.compute(predictions=[pred], references=[[ref]])['score']
        sentence_bleu_scores.append(score)

    df["BLEU_Score"] = sentence_bleu_scores

    worst_cases = df.sort_values(by="BLEU_Score").head(5)
    print("\n--- 5 CÂU CÓ ĐIỂM THẤP NHẤT (Cần phân tích lỗi) ---")
    for idx, row in worst_cases.iterrows():
        print(f"ID: {idx}")
        print(f"Ref: {row['Reference']}")
        print(f"Pred: {row['Prediction']}")
        print(f"BLEU: {row['BLEU_Score']:.2f}")
        print("-" * 20)

    best_cases = df.sort_values(by="BLEU_Score", ascending=False).head(5)
    print("\n--- 5 CÂU CÓ ĐIỂM CAO NHẤT ---")
    for idx, row in best_cases.iterrows():
        print(f"Ref: {row['Reference']}")
        print(f"Pred: {row['Prediction']}")
        print(f"BLEU: {row['BLEU_Score']:.2f}")
        print("-" * 20)
        
    df.to_excel("vlsp_result_optimized_with_scores.xlsx", index=False)
    print("Đã xong! File phân tích: vlsp_result_optimized_with_scores.xlsx")

except FileNotFoundError:
    print("LỖI: Không tìm thấy file 'vlsp_result_optimized.xlsx'. Vui lòng làm theo Cách 2.")

>>> Đã tìm thấy file kết quả cũ. Đang load dữ liệu...
>>> Đang load lại thư viện đánh giá...
>>> Đang thực hiện phân tích lỗi chi tiết...

--- 5 CÂU CÓ ĐIỂM THẤP NHẤT (Cần phân tích lỗi) ---
ID: 2579
Ref: Bảng Các u nội tiết tuyến tuỵ Khối u Hormon Vị trí khối u Triệu chứng và Dấu hiệu U tiết ACTH ACTH Tuyến tuỵ Hội chứng Cushing U tiết gastrin Gastrin Tuỵ (60%) Tá tràng (30%) Khác (10%) Đau bụng, loét dạ dày tá tràng, tiêu chảy U tiết glucagon Glucagon Tuyến tuỵ Không dung nạp glucose, phát ban, gầy sút cân, thiếu máu. U tiết yếu tố giải phóng hormon tăng trưởng Yếu tố giải phóng hormon tăng trưởng Phổi (54%) Tuỵ (30%) hỗng tràng (7%) Khác (13%) Bệnh to đầu chi insulinoma Insulin Tuyến tuỵ Hạ đường huyết khi đói U tăng tiết somatostatin Somatostatin Tuỵ (56%) Tá tràng / hỗng tràng (44%) Không dung nạp glucose, tiêu chảy, sỏi mật Vipoma Peptid vận mạch ruột Tuỵ (90%) Khác (10%) Tiêu chảy mất nước nặng, hạ kali máu, bốc hoả Điều trị khối u nội tiết tuyến tuỵ Phẫu thuật cắt bỏ Điều trị c